# Estrazione Sintesi Fiscale

Questo notebook estrae i campi `CODICE_FISCALE_BENEFICIARIO`, `ELEMENTO_DI_AIUTO_TOTALE`, `IMPORTO_NOMINALE_TOTALE` e `ANNO` da tutti i dataset analizzati (i file *classified_multiclass_aiuti_*), rinominando le prime due colonne in `CODICE_FISCALE`, `IMPORTO_TOTALE` e mantenendo colonna nominale, salvando il risultato aggregato in un nuovo file CSV.

In [6]:
import pandas as pd
import glob
import os

In [7]:
DATA_DIR = '../data'
FILE_PATTERN = 'classified_multiclass_aiuti_*.csv'
OUTPUT_FILE = '../data/export/dataset_importi_cf.csv'

# Definiamo la mappa per estrarre e rinominare contemporaneamente
usecols_map = {
    'CODICE_FISCALE_BENEFICIARIO': 'CODICE_FISCALE',
    'ELEMENTO_DI_AIUTO_TOTALE': 'IMPORTO_TOTALE',
    'IMPORTO_NOMINALE_TOTALE': 'IMPORTO_NOMINALE_TOTALE',
    'ANNO': 'ANNO'
}

files = glob.glob(os.path.join(DATA_DIR, FILE_PATTERN))
print(f"Trovati {len(files)} file per l'estrazione.")

Trovati 12 file per l'estrazione.


In [8]:
df_list = []

for f in files:
    print(f"Elaboro {os.path.basename(f)}...")
    # Leggiamo solo le colonne strettamente necessarie
    df_chunk = pd.read_csv(
        f,
        usecols=list(usecols_map.keys()),
        dtype={'CODICE_FISCALE_BENEFICIARIO': str, 'ANNO': str}  # per mantenere 0 iniziali dei C.F.
    )
    
    # Rinominiamo le colonne per ottenere [CODICE_FISCALE, IMPORTO_TOTALE, IMPORTO_NOMINALE_TOTALE, ANNO]
    df_chunk.rename(columns=usecols_map, inplace=True)
    df_list.append(df_chunk)

Elaboro classified_multiclass_aiuti_2016.csv...
Elaboro classified_multiclass_aiuti_2017.csv...
Elaboro classified_multiclass_aiuti_2015.csv...
Elaboro classified_multiclass_aiuti_2014.csv...
Elaboro classified_multiclass_aiuti_2025.csv...
Elaboro classified_multiclass_aiuti_2019.csv...
Elaboro classified_multiclass_aiuti_2018.csv...
Elaboro classified_multiclass_aiuti_2024.csv...
Elaboro classified_multiclass_aiuti_2023.csv...
Elaboro classified_multiclass_aiuti_2022.csv...
Elaboro classified_multiclass_aiuti_2020.csv...
Elaboro classified_multiclass_aiuti_2021.csv...


In [9]:
if df_list:
    final_df = pd.concat(df_list, ignore_index=True)
    print(f"\nEstrazione completata. Totale record: {len(final_df):,}")
    
    # (Opzionale) assicura che gli importi siano numerici
    final_df['IMPORTO_TOTALE'] = pd.to_numeric(final_df['IMPORTO_TOTALE'], errors='coerce')
    final_df['IMPORTO_NOMINALE_TOTALE'] = pd.to_numeric(final_df['IMPORTO_NOMINALE_TOTALE'], errors='coerce')
    
    # Salvataggio
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nFile salvato in: {OUTPUT_FILE}")
else:
    print("Nessun dato da esportare.")


Estrazione completata. Totale record: 23,957,368

File salvato in: ../data/export/dataset_importi_cf.csv
